# Point clouds

**Geometry type:** `point_cloud`

This notebook demonstrates the full lifecycle of a ZVF point cloud store:

1. Generate synthetic data (positions + attributes)
2. Write to a `.zarrvectors` store
3. Open lazily with `ZarrVectorStore`
4. Inspect metadata without loading vertex data
5. Read all vertices
6. Spatial bounding-box query
7. Build a multi-resolution pyramid
8. Read at coarser levels
9. Export to CSV

**Requirements:** `pip install zarr-vectors`


In [1]:
import numpy as np
import tempfile, os
from pathlib import Path

# All example stores are written to a temporary directory
_tmpdir = tempfile.mkdtemp(prefix="zvf_examples_")
STORE = os.path.join(_tmpdir, "scan.zarrvectors")
print("Store path:", STORE)


Store path: /tmp/zvf_examples_b1uwgqrm/scan.zarrvectors


## 1 · Generate synthetic data

In [2]:
rng = np.random.default_rng(42)

N = 200_000
# Positions: 200k points in a 1000³ µm synchrotron scan volume
positions   = rng.uniform(0, 1000, (N, 3)).astype(np.float32)

# Per-vertex attributes
intensity   = rng.uniform(0, 1, N).astype(np.float32)          # absorption value
label       = rng.integers(0, 8, N).astype(np.int32)            # tissue class
confidence  = rng.uniform(0.5, 1, N).astype(np.float32)

print(f"positions : {positions.shape}  dtype={positions.dtype}")
print(f"intensity : {intensity.shape}  range [{intensity.min():.3f}, {intensity.max():.3f}]")
print(f"label     : {label.shape}  classes {np.unique(label)}")


positions : (200000, 3)  dtype=float32
intensity : (200000,)  range [0.000, 1.000]
label     : (200000,)  classes [0 1 2 3 4 5 6 7]


## 2 · Write to ZVF store

In [3]:
from zarr_vectors.types.points import write_points

write_points(
    STORE,
    positions,
    chunk_shape=(200.0, 200.0, 200.0),   # 200³ µm per chunk → 125 chunks
    bin_shape=(50.0, 50.0, 50.0),        # 50³ µm bins → 64 bins per chunk
    attributes={
        "intensity":  intensity,
        "label":      label,
        "confidence": confidence,
    },
)
print("Write complete.")
print("Store contents:")
for p in sorted(Path(STORE).rglob("zarr.json")):
    print(" ", p.relative_to(STORE))


Write complete.
Store contents:


## 3 · Open lazily with `ZarrVectorStore`

No vertex data is loaded at this step — only `.zattrs` and `zarr.json` metadata files.

In [4]:
from zarr_vectors.lazy import ZarrVectorStore

store = ZarrVectorStore(STORE)
print("Opened lazily — no vertex data loaded yet.")


ImportError: cannot import name 'ZarrVectorStore' from 'zarr_vectors.lazy' (/home/andrew/scripts/zarr-vectors-py/zarr_vectors/lazy/__init__.py)

## 4 · Inspect metadata

In [ ]:
print("geometry_type :", store.geometry_type)
print("spatial_dims  :", store.spatial_dims)
print("chunk_shape   :", store.chunk_shape)
print("bin_shape     :", store.bin_shape)        # at level 0
print("levels        :", store.levels)           # [0] — only base level so far
print("bounding_box  :", store.bounding_box)

# Vertex count from metadata — no data read
print(f"\nVertex count (level 0): {store.vertex_count(level=0):,}")


## 5 · Read all vertices

In [ ]:
from zarr_vectors.types.points import read_points

result = read_points(STORE)

print(f"vertex_count : {result['vertex_count']:,}")
print(f"positions    : {result['positions'].shape}  dtype={result['positions'].dtype}")
print(f"intensity    : {result['attributes']['intensity'].shape}")
print(f"label        : {result['attributes']['label'].shape}")
print()
print("First 5 positions:")
print(result["positions"][:5])


## 6 · Spatial bounding-box query

The query targets individual bins — not full chunks — so only the relevant vertex groups are loaded.

In [ ]:
lo = np.array([100.0, 100.0, 100.0])
hi = np.array([300.0, 300.0, 300.0])

bbox_result = read_points(STORE, bbox=(lo, hi))

expected_fraction = ((hi - lo) / 1000).prod()
expected_n = int(N * expected_fraction)

print(f"Query bbox     : {lo} → {hi}")
print(f"Expected ~      {expected_n:,} vertices  ({expected_fraction*100:.1f}% of volume)")
print(f"Returned        {bbox_result['vertex_count']:,} vertices")
print()
# All returned positions should be within the bbox
assert (bbox_result["positions"] >= lo).all(), "Point below lo!"
assert (bbox_result["positions"] <= hi).all(), "Point above hi!"
print("✓ All returned vertices are within the requested bbox")


## 7 · Build a multi-resolution pyramid

In [ ]:
from zarr_vectors.multiresolution.coarsen import build_pyramid

build_pyramid(
    STORE,
    level_configs=[
        {"bin_ratio": (2, 2, 2)},   # level 1: 8× fewer vertices
        {"bin_ratio": (4, 4, 4)},   # level 2: 64× fewer vertices
    ],
    attribute_aggregation={
        "intensity":  "mean",
        "label":      "majority",
        "confidence": "min",
    },
)
print("Pyramid built.")
print(f"Levels now available: {store.levels}")
for lv in store.levels:
    print(f"  Level {lv}: {store.vertex_count(lv):>8,} vertices"
          f"  bin_shape={store.bin_shape_at(lv)}")


## 8 · Read at coarser levels

In [ ]:
for level in store.levels:
    r = read_points(STORE, level=level)
    print(f"Level {level}: {r['vertex_count']:>8,} vertices")

print()
# Coarsest level — useful for quick overview rendering
coarse = read_points(STORE, level=store.levels[-1])
print(f"Coarsest level ({store.levels[-1]}) positions range:")
print("  min:", coarse["positions"].min(axis=0))
print("  max:", coarse["positions"].max(axis=0))


## 9 · Streaming chunks — memory-bounded iteration

In [ ]:
total = 0
for chunk_coord, chunk_data in store.iter_chunks(level=0):
    total += len(chunk_data["positions"])

print(f"Iterated all chunks at level 0: {total:,} vertices total")
print(f"(matches full read: {result['vertex_count']:,})")


## 10 · Export to CSV

In [ ]:
from zarr_vectors.export.csv_points import export_csv

csv_path = os.path.join(_tmpdir, "scan_export.csv")
export_csv(STORE, csv_path)

# Peek at the output
import pandas as pd
df = pd.read_csv(csv_path, nrows=5)
print(df)
print(f"\nFull CSV: {pd.read_csv(csv_path).shape[0]:,} rows")


## 11 · Validate the store

In [ ]:
from zarr_vectors.validate import validate

result_v = validate(STORE, level=3)
print(result_v.summary())


## Summary

| Step | API used |
|------|----------|
| Write | `write_points(path, positions, chunk_shape, bin_shape, attributes)` |
| Lazy open | `ZarrVectorStore(path)` |
| Read all | `read_points(path)` |
| Bbox query | `read_points(path, bbox=(lo, hi))` |
| Pyramid | `build_pyramid(path, level_configs)` |
| Stream chunks | `store.iter_chunks(level)` |
| Export | `export_csv(path, out)` |
| Validate | `validate(path, level=3)` |
